# ADSP 31014 — Statistical Models for Data Science
## Homework 3

**Name:** Obaro Erigha  
**Term:** Autumn 2026

In [4]:
import re
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

df = pd.read_csv('chicago_taxi_trip_sample.csv')  # keep the CSV in the same folder as this notebook
print(df.shape)
df.head()

(100000, 17)


,trip_id,taxi_id,month,day_of_week,hour,pickup_area,dropoff_area,trip_miles,trip_seconds,fare,tips,tips_percent,tolls,extras,trip_total,payment_type,company
0,259f9c40ef9,f9bc93,1,5,0,Other,Other,0.35,151,4.25,0.00,0.000,0.0,1.0,5.25,Cash,City Service
1,0f61207c6d8,271cde,1,5,0,Downtown,Other,0.53,918,8.50,0.00,0.000,0.0,0.0,8.50,Cash,Sun Taxi
2,252b55a174d,a37d54,1,5,0,Downtown,Downtown,1.05,439,9.36,2.14,0.229,0.0,0.0,11.50,Other,Medallion Leasin
3,b2b5c1ccfdd,3c0868,1,5,0,Downtown,Other,6.73,1277,19.75,5.44,0.275,0.0,1.5,27.19,Credit Card,Sun Taxi
4,5edc7fe28fc,d9587f,1,5,0,Downtown,Downtown,0.90,540,6.50,2.00,0.308,0.0,1.0,10.00,Credit Card,Globe Taxi


In [19]:
# hour_type as defined in the lecture notebook (Step 8b)
hour_type = {'Rush Hour': {7, 8, 9, 15, 16, 17, 18},
             'Night':     {0, 1, 2, 3, 4, 5, 6, 21, 22, 23},
             'Other':     {10, 11, 12, 13, 14, 19, 20}}
df['hour_type'] = pd.Series(data='', index=df.index)
for type, hours in hour_type.items():
    df.loc[df['hour'].isin(hours), 'hour_type'] = type

cols = ['trip_seconds', 'fare', 'hour_type', 'pickup_area']
print(df[cols].isna().sum(), '\n')
df = df.dropna(subset=cols).reset_index(drop=True)

print(df['hour_type'].value_counts(), '\n')
print(df['pickup_area'].value_counts())


print('\nhour_type levels  :', sorted(df['hour_type'].unique()))
print('pickup_area levels:', sorted(df['pickup_area'].unique()))
OHARE = [a for a in df['pickup_area'].unique() if 'hare' in a.lower()][0]
print("O'Hare label in data:", repr(OHARE))

trip_seconds    0
fare            0
hour_type       0
pickup_area     0
dtype: int64 

hour_type
Other        42043
Rush Hour    40773
Night        17184
Name: count, dtype: int64 

pickup_area
Downtown    48862
Other       33391
O'Hare      14725
Midway       3022
Name: count, dtype: int64

hour_type levels  : ['Night', 'Other', 'Rush Hour']
pickup_area levels: ['Downtown', 'Midway', "O'Hare", 'Other']
O'Hare label in data: "O'Hare"


In [21]:
def equation(params, response='trip_seconds'):
    terms = []
    for name, b in params.items():
        if name == 'Intercept':
            terms.append(f'{b:.2f}')
            continue
        factors = []
        for part in name.split(':'):
            m = re.match(r'^(.*)\[(?:T\.)?(.*)\]$', part)   # var[T.level]  or  var[level]
            factors.append(f'1{{{m.group(1)}={m.group(2)}}}' if m else part)
        terms.append(f'{b:+.2f}*' + '*'.join(factors))
    return f'{response} = ' + ' '.join(terms)


def contributes(name, target):
    for part in name.split(':'):
        m = re.match(r'^(.*)\[(?:T\.)?(.*)\]$', part)
        if m and target.get(m.group(1)) != m.group(2):
            return False
    return True


def line_from_params(params, target):
    a0 = sum(b for n, b in params.items() if contributes(n, target) and 'fare' not in n.split(':'))
    a1 = sum(b for n, b in params.items() if contributes(n, target) and 'fare' in n.split(':'))
    return a0, a1

## Question 1a — Two categorical inputs (no interaction)

In [22]:
m1a = smf.ols('trip_seconds ~ hour_type + pickup_area', data=df).fit()
print(m1a.summary())

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.286
Model:                            OLS   Adj. R-squared:                  0.286
Method:                 Least Squares   F-statistic:                     8007.
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:30:33   Log-Likelihood:            -8.0586e+05
No. Observations:              100000   AIC:                         1.612e+06
Df Residuals:                   99994   BIC:                         1.612e+06
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                531

In [8]:
new = pd.DataFrame({'hour_type': ['Rush Hour'], 'pickup_area': ['Downtown']})
print('R^2                  :', round(m1a.rsquared, 4))
print('Number of beta params:', len(m1a.params))
print(m1a.params.round(4), '\n')
print(equation(m1a.params), '\n')
print('Predicted trip_seconds (Rush Hour, Downtown):', round(m1a.predict(new).iloc[0], 2))

R^2                  : 0.2859
Number of beta params: 6
Intercept                  531.8501
hour_type[T.Other]         251.3515
hour_type[T.Rush Hour]     395.7969
pickup_area[T.Midway]      928.8725
pickup_area[T.O'Hare]     1380.3576
pickup_area[T.Other]       378.0860
dtype: float64 

trip_seconds = 531.85 +251.35*1{hour_type=Other} +395.80*1{hour_type=Rush Hour} +928.87*1{pickup_area=Midway} +1380.36*1{pickup_area=O'Hare} +378.09*1{pickup_area=Other} 

Predicted trip_seconds (Rush Hour, Downtown): 927.65


### Answer 1a

R^2 = 0.2859

6 beta parameters: 1 intercept + 2 for hour_type (3 levels - 1) + 3 for pickup_area (4 levels - 1).

Fitted model (reference levels: hour_type = Night, pickup_area = Downtown):

y_hat = 531.85 + 251.35 * 1{hour_type=Other} + 395.80 * 1{hour_type=Rush Hour} + 928.87 * 1{pickup_area=Midway} + 1380.36 * 1{pickup_area=O'Hare} + 378.09 * 1{pickup_area=Other}

Prediction for Rush Hour + Downtown: y_hat = 531.85 + 395.80 = 927.65 seconds.

## Question 1b — Categorical interaction

In [23]:
m1b = smf.ols('trip_seconds ~ hour_type * pickup_area', data=df).fit()
print(m1b.summary())
print('R^2:', round(m1b.rsquared, 4))
print('Number of beta params:', len(m1b.params))

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.308
Model:                            OLS   Adj. R-squared:                  0.308
Method:                 Least Squares   F-statistic:                     4050.
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:34:01   Log-Likelihood:            -8.0427e+05
No. Observations:              100000   AIC:                         1.609e+06
Df Residuals:                   99988   BIC:                         1.609e+06
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                                   coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

In [24]:
target = {'hour_type': 'Rush Hour', 'pickup_area': OHARE}
new = pd.DataFrame({'hour_type': ['Rush Hour'], 'pickup_area': [OHARE]})
print("Predicted trip_seconds (Rush Hour, O'Hare):", round(m1b.predict(new).iloc[0], 2))


names = [n for n in m1b.params.index if contributes(n, target)]
print(m1b.params[names].round(4))
print('Sum of those betas =', round(m1b.params[names].sum(), 2))

Predicted trip_seconds (Rush Hour, O'Hare): 2687.73
Intercept                                        722.2515
hour_type[T.Rush Hour]                           126.4735
pickup_area[T.O'Hare]                            758.5981
hour_type[T.Rush Hour]:pickup_area[T.O'Hare]    1080.4030
dtype: float64
Sum of those betas = 2687.73


In [25]:
grp = df.groupby(['hour_type', 'pickup_area'])['trip_seconds'].agg(['mean', 'count']).reset_index()

print(df.pivot_table(index='hour_type', columns='pickup_area', values='trip_seconds', aggfunc='mean').round(2))

imin = grp['mean'].idxmin()
print('\nSmallest sample mean:')
print(grp.loc[imin])

pickup_area  Downtown   Midway   O'Hare    Other
hour_type                                       
Night          722.25  1356.93  1480.85   927.43
Other          799.68  1698.89  2070.17  1181.03
Rush Hour      848.73  1987.21  2687.73  1276.88

Smallest sample mean:
hour_type           Night
pickup_area      Downtown
mean           722.251543
count                7128
Name: 0, dtype: object


In [26]:
cells = grp[['hour_type', 'pickup_area']].copy()
cells['fitted'] = m1b.predict(cells)
cmp = cells.merge(grp, on=['hour_type', 'pickup_area'])
cmp['abs_diff'] = (cmp['fitted'] - cmp['mean']).abs()
print(cmp.round(2))
print('Max |fitted - sample mean| :', cmp['abs_diff'].max())
print('Intercept vs mean(Night, Downtown):', m1b.params['Intercept'],
      grp.query("hour_type == 'Night' and pickup_area == 'Downtown'")['mean'].iloc[0])

    hour_type pickup_area   fitted     mean  count  abs_diff
0       Night    Downtown   722.25   722.25   7128       0.0
1       Night      Midway  1356.93  1356.93    972       0.0
2       Night      O'Hare  1480.85  1480.85   3153       0.0
3       Night       Other   927.43   927.43   5931       0.0
4       Other    Downtown   799.68   799.68  20299       0.0
5       Other      Midway  1698.89  1698.89   1161       0.0
6       Other      O'Hare  2070.17  2070.17   6413       0.0
7       Other       Other  1181.03  1181.03  14170       0.0
8   Rush Hour    Downtown   848.73   848.73  21435       0.0
9   Rush Hour      Midway  1987.21  1987.21    889       0.0
10  Rush Hour      O'Hare  2687.73  2687.73   5159       0.0
11  Rush Hour       Other  1276.88  1276.88  13290       0.0
Max |fitted - sample mean| : 1.4620127330999821e-10
Intercept vs mean(Night, Downtown): 722.251543209909 722.2515432098766


### Answer 1b

R^2 = 0.3082

12 beta parameters.

Prediction for Rush Hour + O'Hare: 2687.73 seconds (= 722.25 + 126.47 + 758.60 + 1080.40, the intercept plus every coefficient that applies to that cell).

Sample means of trip_seconds (rows = hour_type, columns = pickup_area):

hour_type | Downtown | Midway | O'Hare | Other
Night | 722.25 | 1356.93 | 1480.85 | 927.43
Other | 799.68 | 1698.89 | 2070.17 | 1181.03
Rush Hour | 848.73 | 1987.21 | 2687.73 | 1276.88

Smallest sample mean: hour_type = Night and pickup_area = Downtown, mean = 722.25 seconds (7,128 trips).

Relationship between sample means and betas: TYhe fitted value in each cell is exactly the sample mean of that cell 

12 makes sense because, hour_type has 3 levels and pickup_area has 4 levels, so there are 3 * 4 = 12 cells and the model needs one mean per cell. 
Parameter count = 1 + (3 - 1) + (4 - 1) + (3 - 1)(4 - 1) = 1 + 2 + 3 + 6 = 12.

## Question 1c — Include fare

In [13]:
f1c = 'trip_seconds ~ hour_type * pickup_area + fare:(hour_type * pickup_area)'
m1c = smf.ols(f1c, data=df).fit()
print(m1c.summary())
print('R^2                  :', round(m1c.rsquared, 4))
print('Number of beta params:', len(m1c.params))

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.733
Model:                            OLS   Adj. R-squared:                  0.733
Method:                 Least Squares   F-statistic:                 1.194e+04
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:28:53   Log-Likelihood:            -7.5665e+05
No. Observations:              100000   AIC:                         1.513e+06
Df Residuals:                   99976   BIC:                         1.514e+06
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [27]:
target = {'hour_type': 'Rush Hour', 'pickup_area': OHARE}
p = m1c.params
a0_names = [n for n in p.index if contributes(n, target) and 'fare' not in n.split(':')]
a1_names = [n for n in p.index if contributes(n, target) and 'fare' in n.split(':')]
print('Intercept terms used:', a0_names)
print('Slope terms used    :', a1_names)

a0, a1 = line_from_params(p, target)
print(f'\na0 = {a0:.4f}')
print(f'a1 = {a1:.4f}')

nd = pd.DataFrame({'hour_type': ['Rush Hour'] * 2, 'pickup_area': [OHARE] * 2, 'fare': [0.0, 1.0]})
yhat = m1c.predict(nd).values
print(f'\nprediction at fare=0        : {yhat[0]:.4f}')
print(f'prediction(1) - prediction(0): {yhat[1] - yhat[0]:.4f}')

Intercept terms used: ['Intercept', 'hour_type[T.Rush Hour]', "pickup_area[T.O'Hare]", "hour_type[T.Rush Hour]:pickup_area[T.O'Hare]"]
Slope terms used    : ['fare:hour_type[Rush Hour]', "fare:pickup_area[T.O'Hare]", "fare:hour_type[T.Rush Hour]:pickup_area[T.O'Hare]"]

a0 = 131.5981
a1 = 62.4603

prediction at fare=0        : 131.5981
prediction(1) - prediction(0): 62.4603


In [28]:
sub = df[(df['hour_type'] == 'Rush Hour') & (df['pickup_area'] == OHARE)]
m_sub = smf.ols('trip_seconds ~ fare', data=sub).fit()
print('n rows in subset:', len(sub))
print(m_sub.summary().tables[1])
print(f'Subset fit: a0 = {m_sub.params["Intercept"]:.4f}, a1 = {m_sub.params["fare"]:.4f}')

n rows in subset: 5159
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    131.5981     41.525      3.169      0.002      50.192     213.004
fare          62.4603      0.974     64.122      0.000      60.551      64.370
Subset fit: a0 = 131.5981, a1 = 62.4603


### Answer 1c
R^2 = 0.7331 - a big jump from 0.3082, so fare is by far the most useful predictor.

24 beta parameters.

24 makes sense because, the model fits a separate straight line y = a0 + a1(fare) for each of the 3 x 4 = 12 (hour_type, pickup_area) combinations. Each line has 2 parameters (intercept and slope), so 12 x 2 = 24.

Rush Hour + O'Hare line. Adding up the coefficients that apply to this cell in the summary:

a0 = Intercept + hour_type[T.Rush Hour] + pickup_area[T.O'Hare] + hour_type[T.Rush Hour]:pickup_area[T.O'Hare] = 131.60

a1 = fare:hour_type[Rush Hour] + fare:pickup_area[T.O'Hare] + fare:hour_type[T.Rush Hour]:pickup_area[T.O'Hare] = 62.46

y_hat = 131.60 + 62.46 (fare)

Verification: a separate regression on the 5,159 rows with Rush Hour and O'Hare gives exactly the same a0 = 131.60 and a1 = 62.46. (Note: because there is no standalone fare term in this formula, patsy codes fare:hour_type with all 3 levels, so the Night slope appears directly as fare:hour_type[Night].)


## Question 1d — Three-way interaction

In [16]:
m1d = smf.ols('trip_seconds ~ hour_type * pickup_area * fare', data=df).fit()
print(m1d.summary())

print('R^2   1d / 1c        :', round(m1d.rsquared, 6), '/', round(m1c.rsquared, 6))
print('# betas 1d / 1c      :', len(m1d.params), '/', len(m1c.params))
print('Same fitted values?  :', np.allclose(m1d.fittedvalues, m1c.fittedvalues))
print('Same residual SS?    :', np.isclose(m1d.ssr, m1c.ssr))

                            OLS Regression Results                            
Dep. Variable:           trip_seconds   R-squared:                       0.733
Model:                            OLS   Adj. R-squared:                  0.733
Method:                 Least Squares   F-statistic:                 1.194e+04
Date:                Fri, 23 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:28:53   Log-Likelihood:            -7.5665e+05
No. Observations:              100000   AIC:                         1.513e+06
Df Residuals:                   99976   BIC:                         1.514e+06
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [29]:
rows = []
for h in sorted(df['hour_type'].unique()):
    for a in sorted(df['pickup_area'].unique()):
        t = {'hour_type': h, 'pickup_area': a}
        a0c, a1c = line_from_params(m1c.params, t)
        a0d, a1d = line_from_params(m1d.params, t)
        rows.append((h, a, a0c, a0d, a1c, a1d))
lines = pd.DataFrame(rows, columns=['hour_type', 'pickup_area', 'a0_1c', 'a0_1d', 'a1_1c', 'a1_1d'])
print(lines.round(4))
print('All 12 lines identical?', np.allclose(lines['a0_1c'], lines['a0_1d']) and np.allclose(lines['a1_1c'], lines['a1_1d']))

    hour_type pickup_area     a0_1c     a0_1d    a1_1c    a1_1d
0       Night    Downtown  220.5023  220.5023  33.0445  33.0445
1       Night      Midway  215.2966  215.2966  34.9399  34.9399
2       Night      O'Hare  391.6522  391.6522  28.6863  28.6863
3       Night       Other  246.8445  246.8445  34.9378  34.9378
4       Other    Downtown  122.7785  122.7785  50.4922  50.4922
5       Other      Midway  290.2771  290.2771  41.6844  41.6844
6       Other      O'Hare  193.9208  193.9208  46.3410  46.3410
7       Other       Other  180.0977  180.0977  50.8353  50.8353
8   Rush Hour    Downtown   99.6426   99.6426  56.5154  56.5154
9   Rush Hour      Midway  331.3685  331.3685  48.1178  48.1178
10  Rush Hour      O'Hare  131.5981  131.5981  62.4603  62.4603
11  Rush Hour       Other  187.0286  187.0286  55.0649  55.0649
All 12 lines identical? True


In [30]:
def key(name):
    parts = name.split(':')
    return (tuple(sorted(p for p in parts if p != 'fare')), 'fare' in parts)

c1 = {key(n): b for n, b in m1c.params.items()}
c2 = {key(n): b for n, b in m1d.params.items()}
common = [k for k in c1 if k in c2]
print('Coefficients with identical name and value in 1c and 1d:',
      sum(np.isclose(c1[k], c2[k]) for k in common), 'of', len(c1))

print('\nParameterized differently:')
print('  1c', {n: round(b, 4) for n, b in m1c.params.items() if key(n) not in c2})
print('  1d', {n: round(b, 4) for n, b in m1d.params.items() if key(n) not in c1})

Coefficients with identical name and value in 1c and 1d: 21 of 24

Parameterized differently:
  1c {'fare:hour_type[Night]': 33.0445, 'fare:hour_type[Other]': 50.4922, 'fare:hour_type[Rush Hour]': 56.5154}
  1d {'fare': 33.0445, 'hour_type[T.Other]:fare': 17.4476, 'hour_type[T.Rush Hour]:fare': 23.4709}


### Answer 1d

trip_seconds ~ hour_type * pickup_area * fare expands to all main effects, all two-way interactions and the three-way interaction. 

That is the same model as 1c:

same R^2 = 0.7331, same 24 parameters, identical fitted values and residual sum of squares, and the same 12 fitted straight lines (one per hour_type x pickup_area cell).

only the parameterization differs. 

In 1c there is no standalone fare term, so the three fare:hour_type[] coefficients are the fare slopes for Night / Other / Rush Hour in the reference area (33.04, 50.49, 56.52). 

In 1d those are re-expressed as a baseline fare slope (33.04) plus differences hour_type[T.Other]:fare (17.45) and hour_type[T.Rush Hour]:fare (23.47), e.g. 50.49 = 33.04 + 17.45.

The other 21 coefficients (including all intercept-type terms and the remaining slope interactions) are numerically identical in the two summaries.

So fare:(hour_type * pickup_area) is just a longer way to write the same thing as hour_type * pickup_area * fare; the shorter formula is preferable.

## Question 2 — Asking GenAI about categorical variables with many levels

**Prompt used:** "How should I handle a categorical variable with many levels when fitting a linear regression model?"

1. **Why it is a problem.** A categorical variable with K levels needs K−1 dummy variables. With many levels this inflates the number of parameters, so the model overfits, standard errors become large, some levels have very few observations (unstable or unestimable coefficients), and new/unseen levels at prediction time cannot be handled.
2. **Collapse/group levels.** Merge rare levels into an "Other" category, or group levels using domain knowledge.
3. **Use a numeric summary of the level (encoding).** Replace the category with something meaningful such as frequency/count encoding or target (mean) encoding. Target encoding must be computed within cross-validation folds to avoid leakage.
4. **Regularization.** Fit ridge/lasso/elastic net on the dummy variables so that coefficients of weakly supported levels are shrunk (lasso can set them to zero).
5. **Mixed-effects / hierarchical models.** Treat the variable as a random effect (partial pooling), so levels with little data borrow strength from the overall mean. This also handles unseen levels more gracefully.
6. **Feature selection / testing.** Use an F-test (ANOVA) for the variable as a whole instead of looking at individual dummy t-tests, and use cross-validation to choose the amount of grouping or shrinkage.
7. **Practical tips.** Choose a reference level with plenty of data, make sure each level has enough observations, and avoid high-order interactions between several high-cardinality variables, since the parameter count multiplies
